# P3 Acquisition code: NOAA Climate at a Glance

Skeleton: my P1 loop. Each cell below adds one requirement to it.

| Step | Requirement | Where |
|---|---|---|
| 1 | Honest User-Agent, timeout, non-200 handling, retry | `fetch` |
| 2 | Rate limit (1 request per second or slower) and API headers | `DELAY`, `fetch` |
| 3 | Selector matches nothing (empty or wrong response) | `check` |
| 4 | Cache: never fetch the same resource twice | `get_month` |
| 5 | Pagination: the API returns one month per request | the loop |
| 6 | Fails loudly, with one demonstrated catch | demo cell |

**The P1 skeleton**
```python
for month in range(1, 13):
    stream = req.get(BASE_URL.format(month=month))
    stream.raise_for_status()
    records.update(stream.json()["data"])
```

In [64]:
import json, time
from pathlib import Path
import requests
import pandas as pd

BASE_URL = ("https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/national/"
            "time-series/110/tavg/1/{month}/1978-2026/data.json")
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; DSA405-student-coursework; NC State teaching exercise; contact: pdcurry@ncsu.edu)"}
DELAY = 2            # seconds between real requests (A8 plan; slower than the 1 per second limit)
RAW = Path("data/raw")
RAW.mkdir(parents=True, exist_ok=True)

### 1 and 2, User agent and being nice

In [65]:
def fetch(url, tries=3):
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
        except requests.exceptions.RequestException as e:
            print(f"attempt {attempt}/{tries} failed: {e}")
            time.sleep(DELAY)
            continue
        if r.status_code == 200:
            limits = {k: v for k, v in r.headers.items() if "ratelimit" in k.lower() or k.lower() == "retry-after"}
            print("rate-limit headers:", limits or "none sent")
            return r.json()
        if r.status_code not in (429, 500, 502, 503, 504):
            raise RuntimeError(f"HTTP {r.status_code} for {url}")
        print(f"attempt {attempt}/{tries}: HTTP {r.status_code}, retrying")
        time.sleep(DELAY)
    raise RuntimeError(f"gave up on {url} after {tries} tries")

### 3 chache check

In [66]:
def check(month, out):
    data = out.get("data")
    if not data:
        raise RuntimeError(f"month {month}: response has no data")
    if not all(key.endswith(f"{month:02d}") for key in data):
        raise RuntimeError(f"month {month}: response contains other months")
    if any(v["value"] == -99 for v in data.values()):                    # NOAA's missing-value code
        raise RuntimeError(f"month {month}: contains -99 (missing) values")

In [67]:
def get_month(month):
    path = RAW / f"noaa_tavg_m{month:02d}.json"
    if path.exists():
        print("cached:", path.name)
        return json.loads(path.read_text())
    out = fetch(BASE_URL.format(month=month))
    check(month, out)
    path.write_text(json.dumps(out))
    time.sleep(DELAY)
    return out

### from P1

In [68]:
records = {}
total = 0
for month in range(1, 13):
    data = get_month(month)["data"]
    total += len(data)
    records.update(data)

assert len(records) == total, "months overlapped: update() overwrote something"
print(len(records), "rows,", min(records), "->", max(records))

cached: noaa_tavg_m01.json
cached: noaa_tavg_m02.json
cached: noaa_tavg_m03.json
cached: noaa_tavg_m04.json
cached: noaa_tavg_m05.json
cached: noaa_tavg_m06.json
cached: noaa_tavg_m07.json
cached: noaa_tavg_m08.json
cached: noaa_tavg_m09.json
cached: noaa_tavg_m10.json
cached: noaa_tavg_m11.json
cached: noaa_tavg_m12.json
585 rows, 197801 -> 202609


### fails loudly

Two failures, each caught with a clear message. The second is a real request for a month that does not exist.

In [69]:
try:
    check(1, {"data": {}})
except RuntimeError as e:
    print("FAILED LOUDLY:", e)

before = sorted(p.name for p in RAW.iterdir())
try:
    get_month(13)
except RuntimeError as e:
    print("FAILED LOUDLY:", e)
assert sorted(p.name for p in RAW.iterdir()) == before

FAILED LOUDLY: month 1: response has no data
FAILED LOUDLY: HTTP 404 for https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/national/time-series/110/tavg/1/13/1978-2026/data.json


### Result: same shape the P2 cleaning expects

In [70]:
noaa_raw = pd.DataFrame.from_dict({k: v["value"] for k, v in records.items()}, orient="index", columns=["value"])
noaa_raw.head()

,value
197801,25.86
197901,21.92
198001,31.12
198101,32.41
198201,26.22


| | A8 plan | What the code does | Drift and why |
|---|---|---|---|
| **Price access** | download, "no scraping" | Unchanged.  | none |
| **NOAA access** | REST API | Fetches the data from NOAA| Made the P1 loop 'more freindly' adding user-agent, pause, overlap check.|
| **Request rate** | FRED: 2 s pause. NOAA: only limiting factor is code execution| slowed the loop and added features that stop it from failing quietly.| Rate limits added to API call|
| **Personal data / login / paywall** | none | none: two national monthly aggregates | none |

## Join design

| | |
|---|---|
| **Join key** |first day of the month `datetime64`|
| ** left (NOAA)** | one row per  month |
| ** right (price)** | one row per  month  |
| **Relationship** | one-to-one |
| **Expected rows after join** | 585 |
| **Join type and why** | Left, NOAA on the left. Keeping every NOAA month turns a missing price into a visible NaN|
| **Unmatched rows** | NOAA months with no price row: kept, price = NaN. Price months with no NOAA row: discarded, counted. |


In [71]:
DATA = "https://raw.githubusercontent.com/ErzatsThesaurus/dsa405-project/main/data/raw/"

# NOAA: P2 cleaning
noaa = noaa_raw.copy()
noaa.index = pd.to_datetime(noaa.index, format="%Y%m")
noaa = noaa.sort_index().rename_axis("date").reset_index().rename(columns={"value": "tavg_f"})

# Price: the FRED spreadsheet
price = pd.read_excel(DATA + "APU000072610.xlsx", sheet_name="Monthly")
price = price.rename(columns={"observation_date": "date", "APU000072610": "price_usd_kwh"})
price["date"] = pd.to_datetime(price["date"])

print("NOAA ", len(noaa),  noaa["date"].min().date(),  "->", noaa["date"].max().date())
print("price", len(price), price["date"].min().date(), "->", price["date"].max().date())

NOAA  585 1978-01-01 -> 2026-09-01
price 573 1978-11-01 -> 2026-07-01


In [72]:
n_expected = len(noaa)                                              # left join: same as left table
noaa_no_price = (~noaa["date"].isin(price["date"])).sum()           # NOAA months with no price row
nan_matched = price.loc[price["date"].isin(noaa["date"]), "price_usd_kwh"].isna().sum()
price_only = (~price["date"].isin(noaa["date"])).sum()              # price months that will be dropped
expected_na = noaa_no_price + nan_matched

print("expected rows:", n_expected)
print("expected NaN prices:", expected_na, f"({noaa_no_price} unmatched + {nan_matched} NaN in the file)")
print("price months that will be dropped:", price_only)

expected rows: 585
expected NaN prices: 14 (12 unmatched + 2 NaN in the file)
price months that will be dropped: 0


In [73]:
merged = noaa.merge(price, on="date", how="left", validate="one_to_one", indicator=True)
print(merged.shape)
merged.tail()

(585, 4)


,date,tavg_f,price_usd_kwh,_merge
580,2026-05-01,61.68,0.196,both
581,2026-06-01,70.66,0.198,both
582,2026-07-01,76.91,0.197,both
583,2026-08-01,75.54,NaN,left_only
584,2026-09-01,69.42,NaN,left_only


In [74]:
try:
    noaa.merge(pd.concat([price, price.iloc[[0]]]), on="date", how="left", validate="one_to_one")
except pd.errors.MergeError as e:
    print("MergeError caught:", e)

MergeError caught: Merge keys are not unique in right dataset; not a one-to-one merge


In [75]:
def verify(df, n_expected):
    fails = []

    # Row count. Protects against: fan-out adding rows, or a join silently dropping them.
    if len(df) != n_expected:
        fails.append(f"row count {len(df)} != expected {n_expected}")

    # Value range. Protects against: unit errors (cents vs dollars) and impossible temperatures.
    if not df["tavg_f"].between(15, 85).all():
        fails.append("tavg_f outside 15-85 F")
    if not df["price_usd_kwh"].dropna().between(0.01, 1.00).all():
        fails.append("price_usd_kwh outside $0.01-$1.00 per kWh")

    # Cardinality. Protects against: a month appearing twice, which breaks the one-to-one claim.
    if df["date"].duplicated().any():
        fails.append(f"{df['date'].duplicated().sum()} duplicated month(s)")

    return fails


fails = verify(merged, len(noaa))
print("PASS: all checks held" if not fails else "FAIL:\n  " + "\n  ".join(fails))
assert not fails, fails

PASS: all checks held


###Report one thing that surprised you
FRED has more scraping limits than I had previoulsy realiesed. It does not come up in this project because of how I structured it, but there is a crawl-delay in place there pages.

## P3 Self-Assessment

| # | Criterion | Wt | Score (1-4) |  |
|---|---|---|---|---|
| 1 | Acquisition code | ×2 | 4|  |
| 2 | Ethics & legality in practice | ×2 |4 |
| 3 | Join design | ×2 | 4| |
| 4 | Verification suite | ×3 | 4| |
| 5 | Reproducibility & readability | ×1 | 4|
| |  total | |40 |  |

### AI USE NOTE
Used claude to Generate markdown and help with python.